# Arabic AI vs. Human Text Detection

NLP project: classify Arabic text as **AI-generated (1)** or **human-written (0)** using classical ML, neural networks and AraBERT.

Run the cells in order (Google Colab + Google Drive).

## Part 1 — Data Collection & Merging

In [2]:
import os
import glob
import pandas as pd
import re
from google.colab import drive

# 1. Mount Google Drive
drive.mount("/content/drive")

# 2. Path from your screenshot
folder_path = "/content/drive/MyDrive/Colab Notebooks/AI vs. Human Text"

# 3. List all CSV files
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
total_files_found = len(csv_files)

merged_list = []
merged_count = 0


# Function to check if text is valid Arabic and not corrupted
def is_valid_arabic(text):
    if pd.isna(text) or str(text).strip() == "":
        return False
    # Remove strings that are just question marks
    if re.fullmatch(r"[\?\s]+", str(text)):
        return False
    # Check if it contains at least some Arabic characters
    # This filters out the "Chinese-looking" corrupted text
    arabic_pattern = re.compile(r"[\u0600-\u06FF]")
    if not arabic_pattern.search(str(text)):
        return False
    return True


# 4. Process files
for file in csv_files:
    df = None
    # Try different encodings to solve the "Chinese text" issue
    for enc in ["utf-8-sig", "cp1256", "utf-8", "iso-8859-1"]:
        try:
            df = pd.read_csv(file, encoding=enc)
            break
        except:
            continue

    if df is not None and "text" in df.columns and "label" in df.columns:
        # Keep only text and label
        df_subset = df[["text", "label"]].copy()

        # Apply cleaning: keep only rows with valid Arabic text
        df_subset = df_subset[df_subset["text"].apply(is_valid_arabic)]

        if not df_subset.empty:
            merged_list.append(df_subset)
            merged_count += 1

# 5. Save the cleaned dataset
if merged_list:
    final_df = pd.concat(merged_list, ignore_index=True)

    # Final cleaning: Drop any duplicates that might exist
    final_df.drop_duplicates(subset=["text"], inplace=True)

    output_file = "/content/drive/MyDrive/AI-vs-HUMAN.csv"
    final_df.to_csv(output_file, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 50)
    print("Cleaned Merge Completed!")
    print(f"Total CSV files processed: {total_files_found}")
    print(f"Files contributing clean data: {merged_count}")
    print(f"Total clean rows in final dataset: {len(final_df)}")
    print(f"File saved to: {output_file}")
    print("=" * 50)

Mounted at /content/drive

Cleaned Merge Completed!
Total CSV files processed: 85
Files contributing clean data: 52
Total clean rows in final dataset: 1571
File saved to: /content/drive/MyDrive/AI-vs-HUMAN.csv


## Part 2 — Label Encoding & Arabic Text Preprocessing

In [3]:
# part 2
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords


nltk.download("stopwords")
arabic_stopwords = set(stopwords.words("arabic"))


file_path = "/content/drive/MyDrive/AI-vs-HUMAN.csv"
df = pd.read_csv(file_path)


print("=== [1] Initial Data Exploration ===")
print(f"Dataset Shape (Original): {df.shape}")
print("\nOriginal Label Distribution (Before Cleaning):")
print(df["label"].value_counts())
print("====================================================\n")


print("Cleaning and converting labels to binary (0 and 1)...")

df["label"] = df["label"].astype(str).str.strip().str.lower()
df["label"] = df["label"].str.replace(r"\s+", " ", regex=True)


def map_label_to_numeric(label_val):

    ai_keywords = ["ai", "generated", "artificial"]

    human_keywords = ["human", "humen", "written", "collected", "hand"]

    if any(key in label_val for key in ai_keywords):
        return 1
    elif any(key in label_val for key in human_keywords):
        return 0
    else:
        return None


df["label"] = df["label"].apply(map_label_to_numeric)


if df["label"].isnull().any():
    print(
        "⚠️ تحذير: تم العثور على أسطر تحتوي على ليبل غير مفهوم، سيتم حذفها تلقائياً للحفاظ على دقة الموديل."
    )
    df = df.dropna(subset=["label"])


df["label"] = df["label"].astype(int)


print("\n=== [2] Data Exploration After Label Cleaning ===")
print(f"Dataset Shape (Cleaned): {df.shape}")
print("\nMissing Values per Column:")
print(df.isnull().sum())
print("\nFinal Clean Label Distribution (AI: 1 vs Human: 0):")
print(df["label"].value_counts())
print("====================================================\n")


def preprocess_arabic_text(text):
    if not isinstance(text, str):
        return []

    tashkeel_pattern = re.compile(r"[\u064B-\u0652]")
    text = re.sub(tashkeel_pattern, "", text)

    text = re.sub("[إأآ]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ة", "ه", text)

    text = re.sub(r"[^\u0600-\u06FF\s]", "", text)
    text = re.sub(r"\d+", "", text)

    tokens = text.split()

    filtered_tokens = [
        word for word in tokens if word not in arabic_stopwords and len(word) > 1
    ]

    return filtered_tokens


print("Starting Text Preprocessing... please wait.")


df["tokens"] = df["text"].apply(preprocess_arabic_text)


df["cleaned_text"] = df["tokens"].apply(lambda x: " ".join(x))


output_path = "/content/drive/MyDrive/Preprocessed_NLP_Dataset.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("\n" + "=" * 60)
print("Part 2: Preprocessing Completed Successfully! [cite: 80]")
print(f"Preprocessed data saved to: {output_path}")
print("=" * 60)

print("\nSample Rows after Preprocessing & Label Encoding:")
print(df[["text", "label", "tokens", "cleaned_text"]].head())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


=== [1] Initial Data Exploration ===
Dataset Shape (Original): (1571, 2)

Original Label Distribution (Before Cleaning):
label
AI                     418
human                  334
Human                  312
ai                     192
Ai                      77
human                   31
AI-generated            20
ai                      11
ai generated            10
Humen                   10
hand written            10
Human                   10
HUMAN                   10
ai generated            10
Ai generated            10
0                       10
1                       10
AI_generation           10
AI                      10
human-written           10
AI_generated            10
 humen                   9
collected                9
collect                  5
Human(hand_written)      5
Human(collected)         5
collected                5
Human_written            5
humen                    1
 AI                      1
Al                       1
Name: count, dtype: int64

Cleaning 

In [4]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 33.8 MB/s eta 0:00:00


## Part 3 — Text Representation (BoW, TF-IDF, Word2Vec)

In [5]:
# part 3
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import gensim
import gensim.downloader as api
from gensim.models import Word2Vec


file_path = "/content/drive/MyDrive/Preprocessed_NLP_Dataset.csv"
df = pd.read_csv(file_path)


df["cleaned_text"] = df["cleaned_text"].fillna("")

X_text = df["cleaned_text"].values
y = df["label"].values

print("============================================================")
print("=== Part 3: Text Representation Techniques (Feature Extraction) ===")
print("============================================================\n")


# 1.Bag-of-Words (BoW)
print("1. Extracting Features using Bag-of-Words (BoW)...")

bow_vectorizer = CountVectorizer(max_features=3000)
X_bow = bow_vectorizer.fit_transform(X_text).toarray()
print(f"✅ BoW Matrix Shape: {X_bow.shape} (عدد النصوص, حجم القاموس)")


# TF-IDF

print("\n2. Extracting Features using TF-IDF...")
tfidf_vectorizer = TfidfVectorizer(max_features=3000)
X_tfidf = tfidf_vectorizer.fit_transform(X_text).toarray()
print(f"✅ TF-IDF Matrix Shape: {X_tfidf.shape}")


print("\n3. Loading Pre-trained Arabic Word Embeddings...")


def get_average_word2vec(text, model, vector_size=100):
    words = str(text).split()

    vector_list = [model[word] for word in words if word in model]

    if len(vector_list) == 0:
        return np.zeros(vector_size)
    else:
        return np.mean(vector_list, axis=0)


try:
    print(
        "-> Downloading pre-trained Arabic fastText model (fasttext-wiki-news-subwords-300)..."
    )
    ft_model = api.load("fasttext-wiki-news-subwords-300")
    w2v_vectors = ft_model.wv
    vector_dim = 300
    print("-> Pre-trained Arabic fastText loaded successfully!")
except Exception as e:
    print(f"-> Pre-trained download skipped or failed due to: {e}")
    print(
        "-> Training a Custom Word2Vec model directly on your dataset as an optimal backup..."
    )

    sentences = [str(text).split() for text in X_text]
    backup_model = Word2Vec(
        sentences, vector_size=100, window=5, min_count=1, workers=4
    )
    w2v_vectors = backup_model.wv
    vector_dim = 100
    print("-> Custom Word2Vec trained on your data successfully!")

print("Transforming all text samples into Averaged Word2Vec vectors...")
X_w2v = np.array(
    [get_average_word2vec(text, w2v_vectors, vector_size=vector_dim) for text in X_text]
)
print(f"✅ Word2Vec Matrix Shape: {X_w2v.shape}")


print("\n" + "=" * 60)
print("Structural Comparison for the First Text Sample:")
print("=" * 60)
print(f"Original Text snippet: {X_text[0][:60]}...")
print(
    f"-> Represented by BoW (Non-zero elements count)   : {np.count_nonzero(X_bow[0])} words triggered."
)
print(
    f"-> Represented by TF-IDF (Sample of weights)     : {X_tfidf[0][X_tfidf[0] > 0][:5]}..."
)
print(
    f"-> Represented by Word2Vec (Vector dimension)   : Dense vector with shape {X_w2v[0].shape}"
)
print("=" * 60)
print("Part 3 Completed Successfully! Matrices are ready for Part 4 models.")
print("=" * 60)

=== Part 3: Text Representation Techniques (Feature Extraction) ===

1. Extracting Features using Bag-of-Words (BoW)...
✅ BoW Matrix Shape: (1545, 3000) (عدد النصوص, حجم القاموس)

2. Extracting Features using TF-IDF...
✅ TF-IDF Matrix Shape: (1545, 3000)

3. Loading Pre-trained Arabic Word Embeddings...
-> Downloading pre-trained Arabic fastText model (fasttext-wiki-news-subwords-300)...
[==================================================] 100.0% 958.5/958.4MB downloaded
-> Pre-trained download skipped or failed due to: 'KeyedVectors' object has no attribute 'wv'
-> Training a Custom Word2Vec model directly on your dataset as an optimal backup...
-> Custom Word2Vec trained on your data successfully!
Transforming all text samples into Averaged Word2Vec vectors...
✅ Word2Vec Matrix Shape: (1545, 100)

Structural Comparison for the First Text Sample:
Original Text snippet: يعد الذكاء الاصطناعي ابرز الانجازات حققتها البشريه القرن الح...
-> Represented by BoW (Non-zero elements count)   : 1

## Part 4 — Classical Machine Learning Models

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

results_list = []


def get_classical_models():
    return {
        "Naive Bayes": MultinomialNB(),
        "SVM (RBF)": SVC(kernel="rbf", random_state=42),
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "Random Forest": RandomForestClassifier(random_state=42),
        "AdaBoost": AdaBoostClassifier(random_state=42),
    }


representations = {"Bag-of-Words (BoW)": X_bow, "TF-IDF": X_tfidf, "Word2Vec": X_w2v}

print("============================================================")
print("=== Part 4: Training Classical Machine Learning Models ===")
print("============================================================\n")


for rep_name, X_data in representations.items():
    print(f"🔄 Training all 5 models using: {rep_name}...")

    if rep_name == "Word2Vec":
        scaler = MinMaxScaler()
        X_data_ready = scaler.fit_transform(X_data)
    else:
        X_data_ready = X_data

    X_train, X_test, y_train, y_test = train_test_split(
        X_data_ready, y, test_size=0.2, random_state=42, stratify=y
    )

    models = get_classical_models()

    for model_name, model in models.items():
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        prec, rec, f1, _ = precision_recall_fscore_support(
            y_test, y_pred, average="binary"
        )

        results_list.append(
            {
                "Text Representation": rep_name,
                "Classifier Model": model_name,
                "Accuracy": round(acc, 4),
                "Precision": round(prec, 4),
                "Recall": round(rec, 4),
                "F1-Score": round(f1, 4),
            }
        )
        print(f"   ✔️ {model_name:15} Completed! (Accuracy: {acc:.4f})")
    print("-" * 50)

df_comparison_table = pd.DataFrame(results_list)

print("\n" + "=" * 85)
print("📊 FINAL COMPARATIVE ANALYSIS TABLE FOR REPORT (Part 4):")
print("=" * 85)
df_comparison_table_sorted = df_comparison_table.sort_values(
    by="Accuracy", ascending=False
)
print(df_comparison_table_sorted.to_string(index=False))
print("=" * 85)
print("Part 4 Executed Successfully! copy this table directly to your report.")
print("=" * 85)

=== Part 4: Training Classical Machine Learning Models ===

🔄 Training all 5 models using: Bag-of-Words (BoW)...
   ✔️ Naive Bayes     Completed! (Accuracy: 0.8447)
   ✔️ SVM (RBF)       Completed! (Accuracy: 0.8608)
   ✔️ Decision Tree   Completed! (Accuracy: 0.7346)
   ✔️ Random Forest   Completed! (Accuracy: 0.8544)
   ✔️ AdaBoost        Completed! (Accuracy: 0.7508)
--------------------------------------------------
🔄 Training all 5 models using: TF-IDF...
   ✔️ Naive Bayes     Completed! (Accuracy: 0.8544)
   ✔️ SVM (RBF)       Completed! (Accuracy: 0.8803)
   ✔️ Decision Tree   Completed! (Accuracy: 0.7282)
   ✔️ Random Forest   Completed! (Accuracy: 0.8382)
   ✔️ AdaBoost        Completed! (Accuracy: 0.7605)
--------------------------------------------------
🔄 Training all 5 models using: Word2Vec...
   ✔️ Naive Bayes     Completed! (Accuracy: 0.6149)
   ✔️ SVM (RBF)       Completed! (Accuracy: 0.6926)
   ✔️ Decision Tree   Completed! (Accuracy: 0.6375)
   ✔️ Random Forest   Com

## Part 5 — Feed-Forward Neural Network (Softmax)

In [7]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import pandas as pd
import numpy as np

print("============================================================")
print("=== Part 5: Feed-Forward Neural Network (FNN) with Softmax ===")
print("============================================================\n")


# تعديل الدالة لاستقبال Softmax في طبقة المخرجات
def build_fnn_model_softmax(input_dim):
    model = Sequential(
        [
            Dense(128, activation="relu", input_dim=input_dim),
            Dropout(0.3),
            Dense(64, activation="relu"),
            Dropout(0.2),
            # التعديل الأول: تغيير عدد النيورونات إلى 2 واستخدام activation='softmax'
            Dense(2, activation="softmax"),
        ]
    )

    # التعديل الثاني: تغيير الـ Loss إلى 'sparse_categorical_crossentropy'
    model.compile(
        optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
    )
    return model


fnn_inputs = {"TF-IDF": X_tfidf, "Averaged Word2Vec": X_w2v}

fnn_results = []

for name, X_data in fnn_inputs.items():
    print(f"🧠 Training Softmax FNN using: {name}...")

    X_train, X_test, y_train, y_test = train_test_split(
        X_data, y, test_size=0.2, random_state=42, stratify=y
    )

    # بناء الموديل الجديد
    fnn_model = build_fnn_model_softmax(input_dim=X_data.shape[1])

    # التدريب بنفس الإعدادات السابقة
    fnn_model.fit(
        X_train, y_train, epochs=10, batch_size=32, validation_split=0.1, verbose=1
    )

    # التعديل الثالث: طريقة حساب التوقعات (Prediction)
    # الموديل الآن يخرج مصفوفة بأبعاد (No. of samples, 2) تحتوي على احتمالية الكلاسين معاً
    y_pred_probs = fnn_model.predict(X_test)

    # استخدام np.argmax(axis=1) لاختيار العمود صاحب الاحتمالية الأعلى (0 أو 1) لكل نص
    y_pred = np.argmax(y_pred_probs, axis=1)

    # حساب المقاييس الحاكمة للأداء بعد التعديل
    acc = accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="binary")

    fnn_results.append(
        {
            "Text Representation": name,
            "Classifier Model": "FNN (Softmax Output)",
            "Accuracy": round(acc, 4),
            "Precision": round(prec, 4),
            "Recall": round(rec, 4),
            "F1-Score": round(f1, 4),
        }
    )
    print(f"✅ FNN with {name} (Softmax) finished! (Accuracy: {acc:.4f})\n" + "-" * 50)

# عرض جدول النتائج المحدث بالكامل
df_fnn_results = pd.DataFrame(fnn_results)
print("\n" + "=" * 75)
print("📊 SOFTMAX FNN MODEL PERFORMANCE RESULTS (Part 5):")
print("=" * 75)
print(df_fnn_results.to_string(index=False))
print("=" * 75)

=== Part 5: Feed-Forward Neural Network (FNN) with Softmax ===

🧠 Training Softmax FNN using: TF-IDF...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 20s 319ms/step - accuracy: 0.7131 - loss: 0.6540 - val_accuracy: 0.7581 - val_loss: 0.5875
Epoch 2/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.8741 - loss: 0.4045 - val_accuracy: 0.7903 - val_loss: 0.4289
Epoch 3/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.9362 - loss: 0.2016 - val_accuracy: 0.8226 - val_loss: 0.4211
Epoch 4/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9739 - loss: 0.0982 - val_accuracy: 0.8065 - val_loss: 0.4597
Epoch 5/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9883 - loss: 0.0523 - val_accuracy: 0.7823 - val_loss: 0.5318
Epoch 6/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.9955 - loss: 0.0236 - val_accuracy: 0.7984 - val_loss: 0.5638
Epoch 7/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9973 - loss: 0.0139 - val_accuracy: 0.8065 - val_loss: 0.5960
Epoch 8/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9982 - loss: 0.0095 - val_accuracy: 0.8145 - 

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 17s 193ms/step - accuracy: 0.5369 - loss: 0.6910 - val_accuracy: 0.5161 - val_loss: 0.6780
Epoch 2/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.5737 - loss: 0.6742 - val_accuracy: 0.5806 - val_loss: 0.6708
Epoch 3/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5773 - loss: 0.6718 - val_accuracy: 0.5806 - val_loss: 0.6621
Epoch 4/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6052 - loss: 0.6635 - val_accuracy: 0.6129 - val_loss: 0.6540
Epoch 5/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.6232 - loss: 0.6479 - val_accuracy: 0.6290 - val_loss: 0.6565
Epoch 6/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.6169 - loss: 0.6441 - val_accuracy: 0.6290 - val_loss: 0.6358
Epoch 7/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6466 - loss: 0.6339 - val_accuracy: 0.7016 - val_loss: 0.6216
Epoch 8/10
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.6691 - loss: 0.6235 - val_accuracy: 0.6855 - v

## Part 6 — BiLSTM (Softmax)

In [8]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Embedding,
    LSTM,
    Bidirectional,
    Dense,
    Dropout,
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import pandas as pd
import numpy as np

print("============================================================")
print("=== Part 6: Recurrent Neural Network (BiLSTM) with Softmax ===")
print("============================================================\n")

MAX_VOCAB_SIZE = 10000
MAX_SEQUENCE_LENGTH = 150
print("Tokenizing and Padding sequences for LSTM...")
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_text)

sequences = tokenizer.texts_to_sequences(X_text)

X_lstm = pad_sequences(
    sequences, maxlen=MAX_SEQUENCE_LENGTH, padding="post", truncating="post"
)
print(f"-> LSTM Input Shape: {X_lstm.shape} (عدد العينات, طول النص الموحد)")

X_train_lstm, X_test_lstm, y_train_lstm, y_test_lstm = train_test_split(
    X_lstm, y, test_size=0.2, random_state=42, stratify=y
)


def build_bilstm_model_softmax(vocab_size, embedding_dim=128):
    model = Sequential(
        [
            Input(shape=(MAX_SEQUENCE_LENGTH,)),
            Embedding(input_dim=vocab_size, output_dim=embedding_dim),
            Bidirectional(LSTM(64, return_sequences=False)),
            Dropout(0.3),
            Dense(32, activation="relu"),
            Dropout(0.2),
            Dense(2, activation="softmax"),
        ]
    )

    model.compile(
        optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
    )
    return model


print("\n🧠 Compiling and Training the BiLSTM network...")
# بناء الموديل المحدث
bilstm_model = build_bilstm_model_softmax(vocab_size=MAX_VOCAB_SIZE)

history = bilstm_model.fit(
    X_train_lstm, y_train_lstm, epochs=5, batch_size=32, validation_split=0.1, verbose=1
)

print("\nEvaluating BiLSTM Model...")
# حساب التوقعات بناءً على المخرجات الجديدة
y_pred_probs = bilstm_model.predict(X_test_lstm)


y_pred_lstm = np.argmax(y_pred_probs, axis=1)

# حساب المقاييس
acc_lstm = accuracy_score(y_test_lstm, y_pred_lstm)
prec_lstm, rec_lstm, f1_lstm, _ = precision_recall_fscore_support(
    y_test_lstm, y_pred_lstm, average="binary"
)

# عرض جدول النتائج النهائي المعدل لـ Part 6
lstm_results = [
    {
        "Text Representation": "Word Sequences (Tokenized & Padded)",
        "Classifier Model": "BiLSTM (Softmax Output)",
        "Accuracy": round(acc_lstm, 4),
        "Precision": round(prec_lstm, 4),
        "Recall": round(rec_lstm, 4),
        "F1-Score": round(f1_lstm, 4),
    }
]

df_lstm_results = pd.DataFrame(lstm_results)
print("\n" + "=" * 75)
print("📊 LSTM MODEL PERFORMANCE RESULTS (Part 6):")
print("=" * 75)
print(df_lstm_results.to_string(index=False))
print("=" * 75)
print("All mandatory project parts are now completed and ready!")
print("===========================================================")

=== Part 6: Recurrent Neural Network (BiLSTM) ===

Tokenizing and Padding sequences for LSTM...
-> LSTM Input Shape: (1545, 150) (عدد العينات, طول النص الموحد)

🧠 Compiling and Training the BiLSTM network...
Epoch 1/5
35/35 ━━━━━━━━━━━━━━━━━━━━ 16s 49ms/step - accuracy: 0.6142 - loss: 0.6586 - val_accuracy: 0.6774 - val_loss: 0.5816
Epoch 2/5
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.8624 - loss: 0.3728 - val_accuracy: 0.7984 - val_loss: 0.6540
Epoch 3/5
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.9451 - loss: 0.1615 - val_accuracy: 0.7984 - val_loss: 0.5628
Epoch 4/5
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.9829 - loss: 0.0538 - val_accuracy: 0.7823 - val_loss: 0.7505
Epoch 5/5
35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9964 - loss: 0.0242 - val_accuracy: 0.7903 - val_loss: 0.6505

Evaluating BiLSTM Model...
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step

📊 LSTM MODEL PERFORMANCE RESULTS (Part 6):
                Text Representation       Classif

## Part 7 — SVM Hyperparameter Tuning (GridSearchCV)

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

X_train_tfidf, X_test_tfidf, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

print("--- Starting Hyperparameter Tuning for SVM (TF-IDF Features) ---")

param_grid = {"C": [0.1, 1, 10], "gamma": ["scale", "auto", 0.1]}

svm_base = SVC(kernel="rbf", random_state=42)

grid_search = GridSearchCV(
    estimator=svm_base,
    param_grid=param_grid,
    cv=3,
    scoring="accuracy",
    verbose=2,
    n_jobs=-1,
)

grid_search.fit(X_train_tfidf, y_train)

print("\n=== Grid Search Results ===")
print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"Best Cross-Validation Accuracy: {grid_search.best_score_:.4f}")

best_svm_model = grid_search.best_estimator_
y_pred_tuned_svm = best_svm_model.predict(X_test_tfidf)

tuned_accuracy = accuracy_score(y_test, y_pred_tuned_svm)
print(f"Tuned SVM Test Accuracy: {tuned_accuracy * 100:.2f}%")
print("\nClassification Report for Tuned SVM:")
print(classification_report(y_test, y_pred_tuned_svm))

--- Starting Hyperparameter Tuning for SVM (TF-IDF Features) ---
Fitting 3 folds for each of 9 candidates, totalling 27 fits

=== Grid Search Results ===
Best Hyperparameters: {'C': 10, 'gamma': 'scale'}
Best Cross-Validation Accuracy: 0.8544
Tuned SVM Test Accuracy: 88.35%

Classification Report for Tuned SVM:
              precision    recall  f1-score   support

           0       0.91      0.85      0.88       153
           1       0.86      0.92      0.89       156

    accuracy                           0.88       309
   macro avg       0.89      0.88      0.88       309
weighted avg       0.89      0.88      0.88       309



In [10]:
!pip install transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.8 MB/s eta 0:00:00


## Part 8 — AraBERT Fine-Tuning

In [11]:
import torch
import numpy as np
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import evaluate

X_train_txt, X_test_txt, y_train_txt, y_test_txt = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)

train_dataset = Dataset.from_dict({"text": X_train_txt, "label": y_train_txt})
test_dataset = Dataset.from_dict({"text": X_test_txt, "label": y_test_txt})

model_checkpoint = "aubmindlab/bert-base-arabertv02"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)


def tokenize_function(examples):
    return tokenizer(
        examples["text"], padding="max_length", truncation=True, max_length=128
    )


tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint, num_labels=2
)

metric = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)


training_args = TrainingArguments(
    output_dir="./results_arabert",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

print("--- Starting AraBERT Fine-Tuning ---")
trainer.train()

print("\n--- Final AraBERT Evaluation Results ---")
eval_results = trainer.evaluate()
print(f"AraBERT Test Accuracy: {eval_results['eval_accuracy'] * 100:.2f}%")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/381 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/825k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.64M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


--- Final AraBERT Evaluation Results ---


AraBERT Test Accuracy: 88.67%


## Part 9 — Final Model Comparison

In [14]:
import pandas as pd

# تجميع البيانات واستدعاء نسب الدقة تلقائياً من المتغيرات المحسوبة سابقاً في الكود
comparison_data = {
    "Model Architecture": [
        "Baseline SVM (RBF)",
        "Tuned SVM (GridSearch)",
        "Feed-Forward Neural Network (FNN)",  # تأكد من وجود متغير دقة الـ FNN في كودك السابق
        "Bidirectional LSTM (BiLSTM)",
        "AraBERT (Hugging Face)",
    ],
    "Feature Extraction": [
        "TF-IDF",
        "TF-IDF (Optimized Parameters)",
        "TF-IDF / Dense Embeddings",
        "Custom Embedding Layer",
        "AraBERT Native Tokenizer",
    ],
    "Test Accuracy (%)": [
        # نقوم بضرب القيمة في 100 لتحويلها إلى نسبة مئوية وتقريبها لخانين عشريتين
        round(
            accuracy_score(y_test, y_pred_svm) * 100, 2
        ),  # استدعاء من الـ Baseline SVM
        round(tuned_accuracy * 100, 2),  # استدعاء من الـ Tuned SVM
        round(acc_fnn * 100, 2),  # استدعاء من الـ FNN (حسب اسم المتغير عندك)
        round(acc_lstm * 100, 2),  # استدعاء من الـ BiLSTM
        round(eval_results["eval_accuracy"] * 100, 2),  # استدعاء من نتائج تقييم AraBERT
    ],
}

# تحويل البيانات إلى جدول Pandas DataFrame
df_final_comparison = pd.DataFrame(comparison_data)

# ترتيب الجدول تنازلياً من الأعلى دقة إلى الأقل دقة تلقائياً
df_final_comparison = df_final_comparison.sort_values(
    by="Test Accuracy (%)", ascending=False
)

# عرض النتائج بشكل منسق ونظيف
print("\n" + "=" * 65)
print("🏆 FINAL PROJECT PERFORMANCE COMPARISON (Automated Fetch) 🏆")
print("=" * 65)
print(df_final_comparison.to_string(index=False))
print("=" * 65)

NameError: name 'y_pred_svm' is not defined